In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

#Loading the master merged dataset
df = pd.read_csv("../data/gme_merged_analysis.csv")

#Converting standardized lowercase 'date' column to datetime format
df['date'] = pd.to_datetime(df['date'])

#Sorting chronologically from past to present
df = df.sort_values('date').reset_index(drop=True)

print(f"Dataset loaded successfully with {df.shape[0]} days of trading records!")
df.head()

Dataset loaded successfully with 29 days of trading records!


,date,Close,High,Low,Open,Volume,mean_sentiment,total_posts
0,2021-01-04,4.3125,4.7750,4.2875,4.7500,40090000,0.062833,76
1,2021-01-05,4.3425,4.5200,4.3075,4.3375,19846000,0.124283,60
2,2021-01-06,4.5900,4.7450,4.3325,4.3350,24224800,0.124553,49
3,2021-01-07,4.5200,4.8625,4.5050,4.6175,24517200,0.033894,31
4,2021-01-08,4.4225,4.5750,4.2700,4.5450,25928000,0.122855,55


In [9]:
#Target (Today's direction based on today vs yesterday close)
df['Price_Change'] = df['Close'].diff()
df['Price_Direction'] = (df['Price_Change'] > 0).astype(int)

#Feature engineering (Lagged by 1 day)
df['Yesterday_Posts'] = df['total_posts'].shift(1)
df['Yesterday_Sentiment'] = df['mean_sentiment'].shift(1)

# New Feature A: Yesterday's actual market trading volume
df['Yesterday_Stock_Volume'] = df['Volume'].shift(1)

# New Feature B: Price Momentum (Yesterday's close price vs 2 days ago)
df['Yesterday_Price_Return'] = df['Close'].shift(1) - df['Close'].shift(2)

# New Feature C: 3-Day Rolling Average of Reddit Hype
df['Reddit_3Day_Hype_Avg'] = df['total_posts'].shift(1).rolling(window=3).mean()

df_ml = df.dropna().copy()
print("Advanced feature engineering completed!")
df_ml[['date', 'Price_Direction', 'Yesterday_Posts', 'Yesterday_Sentiment', 'Yesterday_Stock_Volume', 'Yesterday_Price_Return']].head()

Advanced feature engineering completed!


,date,Price_Direction,Yesterday_Posts,Yesterday_Sentiment,Yesterday_Stock_Volume,Yesterday_Price_Return
3,2021-01-07,0,49.0,0.124553,24224800.0,0.2475
4,2021-01-08,0,31.0,0.033894,24517200.0,-0.0700
5,2021-01-11,1,55.0,0.122855,25928000.0,-0.0975
6,2021-01-12,1,106.0,0.076361,59632000.0,0.5625
7,2021-01-13,1,50.0,0.070948,28242800.0,0.0025


In [12]:
feature_cols = [
    'Yesterday_Posts', 
    'Yesterday_Sentiment', 
    'Yesterday_Stock_Volume', 
    'Yesterday_Price_Return',
    'Reddit_3Day_Hype_Avg'
]

X = df_ml[feature_cols]
y = df_ml['Price_Direction']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, shuffle=False)
print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")

Training features shape: (20, 5)
Testing features shape: (6, 5)


In [13]:
model = RandomForestClassifier(
    n_estimators=100, 
    max_depth=3,           
    min_samples_split=4,    
    random_state=42
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"--- Updated Model Accuracy Score: {accuracy:.2%} ---\n")
print("Updated Classification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

--- Updated Model Accuracy Score: 66.67% ---

Updated Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.33      0.50         3
           1       0.60      1.00      0.75         3

    accuracy                           0.67         6
   macro avg       0.80      0.67      0.62         6
weighted avg       0.80      0.67      0.62         6



In [15]:
importances = model.feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance_Score': importances
})
feature_importance_df = feature_importance_df.sort_values(by='Importance_Score', ascending=False).reset_index(drop=True)
print("---Feature Importances ---")
print(feature_importance_df)

---Feature Importances ---
                  Feature  Importance_Score
0         Yesterday_Posts          0.253187
1  Yesterday_Stock_Volume          0.208614
2    Reddit_3Day_Hype_Avg          0.208550
3  Yesterday_Price_Return          0.169335
4     Yesterday_Sentiment          0.160314
